<a href="https://colab.research.google.com/github/6hamuge/TRO-SORT/blob/repro%2Foa-sort/notebooks/OA_SORT_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OA-SORT 재구현 · Google Colab

공식 **Hybrid-SORT** 코드에 OA-SORT 논문의 **OAM, OAO, BAM**을 연결했다. 공개 YOLOX 가중치로 프레임별 검출을 한 번 실행하고, 기본 Hybrid-SORT와 OA-SORT가 같은 검출을 읽어 사람 ID를 추적한다. 추가 학습은 필요 없다.

1. [Google Colab](https://colab.research.google.com/)에서 **파일 → 노트북 업로드**로 이 파일을 연다.
2. **런타임 → 런타임 유형 변경 → T4 GPU**를 선택한다.
3. 위에서부터 실행한다. 기본 입력은 DanceTrack 0079의 앞 180프레임이다. 내 영상은 첫 셀에서 `INPUT_MODE = "upload_video"`를 선택한다.
4. 마지막 셀에서 MP4, 추적 TXT, 공통 검출과 설정을 ZIP으로 다운로드할 수 있다.

[OA-SORT 논문 §4와 Appendix §7](https://arxiv.org/html/2603.06034v2) · [Hybrid-SORT 공식 README](https://github.com/ymzis69/HybridSORT) · [DanceTrack](https://github.com/DanceTrack/DanceTrack)

## 1. 입력, 가중치와 실행 모드 설정

**이미 다운로드한 가중치가 있으면** `WEIGHTS_MODE = "local"`로 바꾸고 `WEIGHTS_PATH`에 파일 경로를 적는다. Drive 경로를 쓰려면 `MOUNT_DRIVE = True`로 바꾼다. 공개 Drive 파일 링크가 있으면 `"drive_url"` 모드도 쓸 수 있다. `"official"`은 공식 Model Zoo 폴더에서 README가 지정한 파일명을 찾아 내려받는다.

DanceTrack 예제는 ZIP에서 필요한 프레임만 가져오도록 먼저 시도한다. 부분 다운로드가 안 되면 기본 설정에서 공식 validation ZIP 전체(4.21 GB)를 다운로드한다. 이때 직접 영상을 올리고 싶으면 `INPUT_MODE = "upload_video"`로 바꿔 해당 구간을 다시 실행한다.

`RUN_MODES`에서 실행할 모드를 고른다. `"hybrid"`는 비교용 Hybrid-SORT, `"oa_sort"`는 OAM·OAO·BAM, `"oao_only"`와 `"bam_only"`는 각 모듈의 영향을 확인하는 설정이다. `OA_GAUSSIAN = False`는 OAM에서 Gaussian 가중치 없이 가려진 면적 비율을 사용한다.

In [ ]:
from pathlib import Path

INPUT_MODE = "dancetrack"    # "dancetrack", "upload_video", "local_video"
INPUT_VIDEO = ""             # 예: "/content/drive/MyDrive/dance.mp4"
SEQUENCE = "dancetrack0079"
MAX_FRAMES = 0              # 0이면 전체 프레임
DANCETRACK_ZIP_PATH = ""       # 이미 val.zip이 있으면 전체 경로 입력
ALLOW_FULL_ZIP_DOWNLOAD = True
MOUNT_DRIVE = False

WEIGHTS_MODE = "official"     # "official", "local", "drive_url", "bytetrack_hf"
WEIGHTS_PATH = ""             # 예: "/content/drive/MyDrive/ocsort_dance_model.pth.tar"
WEIGHTS_FILE_URL = ""         # drive_url 모드에서 공유된 '파일' 링크 입력
ALLOW_LEGACY_CHECKPOINT = False  # 신뢰하는 원본에서 weights_only 로딩 오류가 날 때만 True

USE_FP16 = True               # README의 DanceTrack 설정
REPO_REF = ""                 # 재실험할 때 출력된 Git 커밋 SHA를 넣을 수 있음
ROOT = Path("/content/hybrid_sort_colab")
REPO = ROOT / "HybridSORT"
VENV = ROOT / "venv"
PYTHON = VENV / "bin/python"
VAL_CFG = "exps/example/mot/yolox_dancetrack_val_hybrid_sort.py"
CHECKPOINT_NAME = "ocsort_dance_model.pth.tar"
MODEL_ZOO_URLS = [
    "https://drive.google.com/drive/folders/18IsZGeGiyKDshhYIzbpYXoNEcBhPY8lN",
    "https://drive.google.com/drive/folders/1LnhZVJlpufUnWuObZASIN1KwfhuvT_a8",
]
VAL_ZIP_URL = "https://huggingface.co/datasets/noahcao/dancetrack/resolve/main/val.zip"
FALLBACK_WEIGHTS_URL = (
    "https://huggingface.co/noahcao/dancetrack_models/resolve/main/"
    "bytetrack_models/bytetrack_model.pth.tar"
)
ROOT.mkdir(parents=True, exist_ok=True)
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if MAX_FRAMES < 0:
    raise ValueError("MAX_FRAMES는 0 이상이어야 합니다.")

RUN_MODES = ("hybrid", "oa_sort")  # 예: ("hybrid", "oao_only", "bam_only", "oa_sort")
OA_TAU = 0.15                    # DanceTrack, Eq. (8)
OA_DEPTH_PIXELS = 5               # 박스의 아래쪽 y 차이, 원본 영상 픽셀
OA_GAUSSIAN = True

## 2. 공식 코드와 실행 환경 설치

저장소 README는 Python 3.8 / PyTorch 1.10으로 테스트했다고 안내한다. 이 노트북은 Colab 커널과 별도인 **Python 3.11 / PyTorch 2.6 CUDA 12.4** 환경에서 실행하고, 예전 NumPy API와 호환되는 버전을 사용한다. 추적 데모에 필요한 패키지를 설치하며 ONNX 내보내기 패키지는 생략한다. 최초 설치에는 시간이 걸릴 수 있다. [공식 설치 안내](https://github.com/ymzis69/HybridSORT#installation) · [PyTorch 버전별 설치 안내](https://docs.pytorch.org/get-started/previous-versions/)

In [ ]:
import os
import shutil
import subprocess
import sys

def run(args, *, cwd=None, env=None, capture=False):
    result = subprocess.run(
        [str(arg) for arg in args], cwd=cwd, env=env, check=True, text=True,
        stdout=subprocess.PIPE if capture else None,
    )
    return result.stdout.strip() if capture else None

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Colab 런타임 유형에서 GPU(T4)를 선택한 후 다시 실행하세요.")
run(["nvidia-smi", "-L"])
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("ffmpeg와 ffprobe가 필요합니다. Colab 기본 Linux 런타임에서 실행하세요.")

if not (REPO / ".git").is_dir():
    run(["git", "clone", "https://github.com/ymzis69/HybridSORT.git", REPO])
if REPO_REF:
    run(["git", "checkout", "--detach", REPO_REF], cwd=REPO)
REVISION = run(["git", "rev-parse", "HEAD"], cwd=REPO, capture=True)
print("Hybrid-SORT commit:", REVISION)

run([sys.executable, "-m", "pip", "install", "-q", "uv", "remotezip==0.12.3"])
uv = shutil.which("uv") or str(Path(sys.executable).with_name("uv"))
bootstrap = VENV / ".hybrid-sort-env-v1"
if not bootstrap.is_file():
    run([uv, "python", "install", "3.11"])
    if not PYTHON.exists():
        run([uv, "venv", "--python", "3.11", VENV])
    install = [uv, "pip", "install", "--python", PYTHON]
    run([*install, "--index-url", "https://download.pytorch.org/whl/cu124",
         "torch==2.6.0", "torchvision==0.21.0"])
    run([*install,
         "numpy==1.23.5", "scipy==1.13.1", "opencv-python-headless==4.10.0.84",
         "lap==0.5.12", "filterpy==1.4.5", "scikit-image==0.22.0",
         "pandas==2.2.3", "matplotlib==3.7.5", "scikit-learn==1.5.2",
         "cython==0.29.37", "setuptools==69.5.1", "wheel",
         "loguru", "tqdm", "pillow", "thop", "ninja", "tabulate", "tensorboard",
         "h5py", "xmltodict", "yacs", "easydict", "termcolor", "fvcore",
         "iopath", "pyyaml", "prettytable", "einops", "faiss-cpu==1.8.0.post1", "gdown==6.1.0"])
    run([*install, "--no-build-isolation", "--no-binary", "cython-bbox",
         "--no-binary", "pycocotools", "cython-bbox==0.1.5", "pycocotools==2.0.7"])
    bootstrap.touch()

BASE_ENV = {**os.environ, "PYTHONPATH": str(REPO), "MPLBACKEND": "Agg",
            "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2", "YOLOX_DATADIR": str(REPO / "datasets")}
run([PYTHON, "-c", "import cv2, numpy, torch, torchvision, lap, cython_bbox; "
     "assert torch.cuda.is_available(), 'CUDA를 사용할 수 없습니다. GPU 런타임을 확인하세요.'; "
     "print('GPU:', torch.cuda.get_device_name()); "
     "print('torch / numpy:', torch.__version__, numpy.__version__)"], cwd=REPO, env=BASE_ENV)

## 3. 학습된 YOLOX 가중치 가져오기

공식 예제의 파일명은 `ocsort_dance_model.pth.tar`다. 이 파일은 영상에서 사람 박스를 검출하는 데 쓰이며, Hybrid-SORT 코드는 검출 박스를 시간 순서대로 연결해 ID를 붙인다. 다운로드가 제한되면 아래 Drive 페이지에서 파일을 직접 받아 Drive에 넣고 1번 셀의 `local` 모드를 사용한다.

선택적으로 `WEIGHTS_MODE = "bytetrack_hf"`를 사용하면 DanceTrack 팀이 배포한 ByteTrack용 YOLOX를 내려받아 실행할 수 있다. 이 경우 검출 가중치가 달라지므로 실험 기록에도 별도로 적는다. 임의의 `.pt` 파일을 넣으려면 먼저 해당 파일이 이 YOLOX 설정과 호환되는지 확인한다.

[Hybrid-SORT 가중치 폴더](https://drive.google.com/drive/folders/18IsZGeGiyKDshhYIzbpYXoNEcBhPY8lN) · [OC-SORT 가중치 폴더](https://drive.google.com/drive/folders/1LnhZVJlpufUnWuObZASIN1KwfhuvT_a8) · [대체 YOLOX 가중치](https://huggingface.co/noahcao/dancetrack_models/tree/main/bytetrack_models)

In [ ]:
import hashlib
import urllib.request

PRETRAINED = REPO / "pretrained"
PRETRAINED.mkdir(exist_ok=True)

def fetch_http(url, target):
    partial = target.with_name(target.name + ".part")
    with urllib.request.urlopen(url, timeout=60) as response, partial.open("wb") as output:
        shutil.copyfileobj(response, output, length=1024 * 1024)
    partial.replace(target)

if WEIGHTS_MODE == "local":
    if not WEIGHTS_PATH:
        raise ValueError("1번 셀에 WEIGHTS_PATH를 지정하세요.")
    checkpoint = Path(WEIGHTS_PATH).expanduser()
elif WEIGHTS_MODE == "bytetrack_hf":
    checkpoint = PRETRAINED / "bytetrack_model.pth.tar"
    if not checkpoint.is_file() or checkpoint.stat().st_size < 1_000_000:
        fetch_http(FALLBACK_WEIGHTS_URL, checkpoint)
elif WEIGHTS_MODE in {"official", "drive_url"}:
    # 직접 지정한 다른 파일 링크는 공식 체크포인트 캐시와 구분한다.
    url_key = hashlib.sha256(WEIGHTS_FILE_URL.encode()).hexdigest()[:12]
    checkpoint = PRETRAINED / (CHECKPOINT_NAME if WEIGHTS_MODE == "official"
                               else f"drive-{url_key}.pth.tar")
    if WEIGHTS_MODE == "drive_url" and not WEIGHTS_FILE_URL:
        raise ValueError("1번 셀에 WEIGHTS_FILE_URL을 지정하세요.")
    if not checkpoint.is_file() or checkpoint.stat().st_size < 1_000_000:
        download_code = r'''
import pathlib, sys
import gdown

target = pathlib.Path(sys.argv[1])
name, direct_url, *folders = sys.argv[2:]
temporary = target.with_name(target.name + ".part")
errors = []
if not direct_url:
    for folder in folders:
        try:
            entries = gdown.download_folder(url=folder, skip_download=True)
            entry = next((item for item in entries if pathlib.Path(item.path).name == name), None)
            if entry is None:
                raise FileNotFoundError(name)
            result = gdown.download(id=entry.id, output=str(temporary))
            if not result or temporary.stat().st_size < 1_000_000:
                raise OSError("가중치 다운로드가 완료되지 않았습니다.")
            temporary.replace(target)
            break
        except Exception as exc:
            errors.append(f"{folder}: {exc}")
    else:
        raise SystemExit("공식 가중치 자동 다운로드 실패. 파일을 직접 받은 뒤 local 모드를 사용하세요.\n"
                         + "\n".join(errors))
else:
    result = gdown.download(url=direct_url, output=str(temporary))
    if not result or temporary.stat().st_size < 1_000_000:
        raise SystemExit("Drive 링크로 가중치를 받지 못했습니다. local 모드를 사용하세요.")
    temporary.replace(target)
'''
        run([PYTHON, "-c", download_code, checkpoint, CHECKPOINT_NAME,
             WEIGHTS_FILE_URL if WEIGHTS_MODE == "drive_url" else "", *MODEL_ZOO_URLS],
            cwd=REPO, env=BASE_ENV)
else:
    raise ValueError(f"지원하지 않는 WEIGHTS_MODE: {WEIGHTS_MODE}")

if not checkpoint.is_file() or checkpoint.stat().st_size < 1_000_000:
    raise FileNotFoundError(f"가중치 파일을 확인하세요: {checkpoint}")
with checkpoint.open("rb") as file:
    checkpoint_sha256 = hashlib.file_digest(file, "sha256").hexdigest()
print(f"YOLOX checkpoint: {checkpoint} ({checkpoint.stat().st_size / 1e6:.1f} MB)")
print("SHA-256:", checkpoint_sha256)

## 4. 추적할 프레임 준비

`dancetrack` 모드는 공식 README와 같은 예제 시퀀스를 사용한다. `upload_video`는 실행할 때 나타나는 창에서 영상 파일 하나를 고른다. `local_video`는 `INPUT_VIDEO`에 적은 Colab 또는 Drive의 영상을 읽는다. 영상도 JPEG 프레임으로 바꿔 README의 `image` 데모로 실행한다.

In [ ]:
from datetime import datetime
from pathlib import PurePosixPath
import re
import zipfile

from remotezip import RemoteZip

RUN_DIR = ROOT / "runs" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
FRAMES_DIR = RUN_DIR / "input_frames"
FRAMES_DIR.mkdir(parents=True)
VAL_CACHED_ZIP = ROOT / "dancetrack_val.zip"
FRAME_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def read_sequence(archive):
    prefix = f"{SEQUENCE}/img1/"
    entries = {}
    for entry in archive.namelist():
        path = PurePosixPath(entry)
        if prefix in entry and path.parent.name == "img1" and path.parent.parent.name == SEQUENCE:
            if path.suffix.lower() in FRAME_EXTENSIONS and path.stem.isdigit():
                entries[path.name] = entry
    if not entries:
        raise FileNotFoundError(f"val.zip에서 {prefix}를 찾지 못했습니다.")
    for name in sorted(entries, key=lambda name: int(Path(name).stem))[:MAX_FRAMES or None]:
        with archive.open(entries[name]) as source, (FRAMES_DIR / name).open("wb") as destination:
            shutil.copyfileobj(source, destination)
    sample_entry = PurePosixPath(next(iter(entries.values())))
    seqinfo_path = str(sample_entry.parent.parent / "seqinfo.ini")
    if seqinfo_path not in archive.namelist():
        return 20.0
    seqinfo = archive.read(seqinfo_path).decode("utf-8-sig")
    match = re.search(r"(?im)^\s*frameRate\s*=\s*([.\d]+)", seqinfo)
    return float(match.group(1)) if match else 20.0

if INPUT_MODE == "dancetrack":
    if not re.fullmatch(r"dancetrack\d{4}", SEQUENCE):
        raise ValueError("시퀀스명은 dancetrack0079 형태여야 합니다.")
    local_zip = Path(DANCETRACK_ZIP_PATH) if DANCETRACK_ZIP_PATH else VAL_CACHED_ZIP
    if DANCETRACK_ZIP_PATH or local_zip.is_file():
        with zipfile.ZipFile(local_zip) as archive:
            fps = read_sequence(archive)
    else:
        try:
            with RemoteZip(VAL_ZIP_URL, support_suffix_range=False, timeout=25) as archive:
                fps = read_sequence(archive)
        except Exception as error:
            for image in FRAMES_DIR.iterdir():
                image.unlink()
            if not ALLOW_FULL_ZIP_DOWNLOAD:
                raise RuntimeError("부분 다운로드 실패. val.zip 경로를 지정하거나 영상 입력을 선택하세요.") from error
            print(f"부분 다운로드 실패({type(error).__name__}). 공식 val.zip 전체(4.21 GB)를 받습니다.",
                  flush=True)
            run(["curl", "-fL", "--retry", "3", "--continue-at", "-", "--output",
                 VAL_CACHED_ZIP.with_suffix(".zip.part"), VAL_ZIP_URL])
            VAL_CACHED_ZIP.with_suffix(".zip.part").replace(VAL_CACHED_ZIP)
            with zipfile.ZipFile(VAL_CACHED_ZIP) as archive:
                fps = read_sequence(archive)
elif INPUT_MODE in {"upload_video", "local_video"}:
    if INPUT_MODE == "upload_video":
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("영상 파일 한 개를 선택하세요.")
        name, data = next(iter(uploaded.items()))
        video = RUN_DIR / Path(name).name
        video.write_bytes(data)
        del uploaded, data
    else:
        if not INPUT_VIDEO:
            raise ValueError("1번 셀에서 INPUT_VIDEO를 지정하세요.")
        video = Path(INPUT_VIDEO)
    if not video.is_file():
        raise FileNotFoundError(video)

    from fractions import Fraction
    rate = run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                "stream=avg_frame_rate", "-of", "default=nw=1:nk=1", video], capture=True)
    fps = float(Fraction(rate)) if rate not in {"", "0/0", "N/A"} else 30.0
    if fps <= 0:
        fps = 30.0
    limit = ["-frames:v", MAX_FRAMES] if MAX_FRAMES else []
    run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", video,
         "-vf", f"fps={fps:.8f}", *limit, "-q:v", "2", FRAMES_DIR / "%08d.jpg"])
else:
    raise ValueError(f"지원하지 않는 INPUT_MODE: {INPUT_MODE}")

input_images = sorted(path for path in FRAMES_DIR.iterdir() if path.suffix.lower() in FRAME_EXTENSIONS)
if not input_images:
    raise RuntimeError("읽은 프레임이 없습니다.")
print(f"입력: {len(input_images)}프레임, {fps:g} FPS")
print("프레임 폴더:", FRAMES_DIR)

## 5. OAM · OAO · BAM 구현

| 모듈 | 적용 시점과 입력 | 구현 |
| --- | --- | --- |
| OAM | 현재 KF 예측 박스, 프레임 끝에 매칭된 실제 관측 박스 | 아래쪽 y 차이로 앞뒤를 판정한다. 교집합 마스크를 합집합으로 만든 뒤 프레임 전체에서 합성한 Gaussian Map으로 가중한다. |
| OAO | 1차 고신뢰 매칭 | 트랙별 예측 가림 계수와 HMIoU를 식 (8)로 결합한다. 기존 방향·신뢰도 항과 후보 임계값을 사용한다. |
| BAM | 2차 저신뢰 매칭 뒤 KF 갱신 | 직전 관측의 가림 계수와 현재 검출의 HMIoU로 식 (9)의 가중치를 계산한다. KF 혁신 벡터에서 위치·면적·종횡비 성분에 가중치를 곱한다. |

재현 기준은 **arXiv v2, §4 및 Algorithm 1–5**다. DanceTrack의 Gaussian 표준편차는 `w/(3√2)`, `h/3`이다. 가림 마스크의 교집합 좌표에는 모두 왼쪽·위쪽 원점을 빼는 방식으로 부록 Algorithm 1의 슬라이스를 옮겼다. 이미지 밖 좌표는 이미지 경계로 자르고, 픽셀 마스크 경계는 내림·올림으로 양자화한다.

OAO는 Hybrid-SORT가 배정 행렬을 푸는 분기에서 공간 점수에 적용된다. 기존 후보가 이미 일대일로 확정되는 분기와 최종 후보 임계값도 사용한다. BAM은 Hybrid-SORT가 별도로 모델링하는 confidence 잔차와 측정 이력을 유지한다. ORU로 궤적을 복구한 프레임에서는 현재 측정의 최종 잔차에만 BAM을 적용한다. 이 세부 설정을 사용한 논문 기반 재구현이므로, 결과를 기록할 때 **OA-SORT (reimplementation)**으로 표기한다.

In [ ]:
# 아래 Python 모듈은 결과 ZIP에도 포함된다.
CODE_DIR = RUN_DIR / "source"
CODE_DIR.mkdir(exist_ok=True)
OA_SOURCE = r'''
"""Paper reimplementation of OA-SORT on top of upstream Hybrid-SORT."""
import ast
from collections import Counter
from dataclasses import dataclass
import inspect
import math
from textwrap import dedent
from types import FunctionType

import numpy as np


@dataclass(frozen=True)
class OAConfig:
    use_oao: bool = True
    use_bam: bool = True
    gaussian: bool = True
    tau: float = 0.15
    depth_pixels: float = 5.0
    kx: float = 3 * math.sqrt(2)
    ky: float = 3.0

    def __post_init__(self):
        if not all(math.isfinite(value) for value in (self.tau, self.depth_pixels, self.kx, self.ky)):
            raise ValueError("OA-SORT parameters must be finite.")
        if not 0 <= self.tau <= 1 or min(self.depth_pixels, self.kx, self.ky) <= 0:
            raise ValueError("Require 0 <= tau <= 1 and positive depth_pixels, kx and ky.")


def occlusion_coefficients(boxes, image_hw, config):
    """OAM (Eq. 2–7). Boxes are N×4 xyxy in original image pixels."""
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    coefficients = np.zeros(len(boxes), dtype=np.float32)
    height, width = map(int, image_hw)
    if len(boxes) < 2 or min(height, width) <= 0:
        return coefficients

    finite = np.isfinite(boxes).all(axis=1)
    # Use floor/ceil for the image slices and floating-point bottom edges for depth.
    clipped = np.clip(np.where(finite[:, None], boxes, 0), 0, [width, height, width, height])
    starts = np.floor(clipped[:, :2]).astype(np.intp)
    ends = np.ceil(clipped[:, 2:]).astype(np.intp)
    valid = finite & (clipped[:, 2:] > clipped[:, :2]).all(axis=1)
    overlap_start = np.maximum(starts[:, None, :], starts[None, :, :])
    overlap_end = np.minimum(ends[:, None, :], ends[None, :, :])
    overlap = (overlap_end > overlap_start).all(axis=2) & valid[:, None] & valid[None, :]
    # Suppress pixel quantization overlaps between geometrically disjoint boxes.
    overlap &= (np.minimum(clipped[:, None, 2:], clipped[None, :, 2:])
                > np.maximum(clipped[:, None, :2], clipped[None, :, :2])).all(axis=2)
    np.fill_diagonal(overlap, False)
    bottoms = clipped[:, 3]
    occludes = overlap & ((bottoms[None, :] - bottoms[:, None]) >= config.depth_pixels)
    blocked = np.flatnonzero(occludes.any(axis=1))
    if not len(blocked):
        return coefficients

    if config.gaussian:
        gaussian_map = np.zeros((height, width), dtype=np.float32)
        for index in np.flatnonzero(overlap.any(axis=1)):
            left, top = starts[index]
            right, bottom = ends[index]
            center = (clipped[index, :2] + clipped[index, 2:]) * 0.5
            sigma = (clipped[index, 2:] - clipped[index, :2]) / [config.kx, config.ky]
            dx = (np.arange(left, right) + 0.5 - center[0]) / sigma[0]
            dy = (np.arange(top, bottom) + 0.5 - center[1]) / sigma[1]
            weights = np.exp(-0.5 * (dy[:, None] ** 2 + dx[None, :] ** 2))
            region = gaussian_map[top:bottom, left:right]
            np.maximum(region, weights, out=region)

    for index in blocked:
        left, top = starts[index]
        right, bottom = ends[index]
        mask = np.zeros((bottom - top, right - left), dtype=bool)
        for other in np.flatnonzero(occludes[index]):
            x1, y1 = overlap_start[index, other]
            x2, y2 = overlap_end[index, other]
            mask[y1 - top:y2 - top, x1 - left:x2 - left] = True
        covered = (gaussian_map[top:bottom, left:right][mask].sum(dtype=np.float64)
                   if config.gaussian else mask.sum())
        coefficients[index] = covered / mask.size
    return coefficients


def occlusion_aware_similarity(similarity, predicted_occlusion, tau):
    """OAO (Eq. 8). Columns represent predicted tracks."""
    return (1 - tau) * similarity + tau * (1 - predicted_occlusion[None, :])


def bias_aware_weight(spatial_similarity, last_observed_occlusion):
    """BAM (Eq. 9), with HMIoU following the paper's Hybrid-SORT experiment."""
    return float(np.clip(spatial_similarity, 0, 1) * (1 - last_observed_occlusion))


def damp_geometry_residual(residual, weight):
    # Hybrid-SORT measurement layout: x, y, area, confidence, aspect ratio.
    if residual.shape[0] != 5:
        raise RuntimeError(f"Expected the 5D Hybrid-SORT measurement, got {residual.shape}.")
    residual[[0, 1, 2, 4]] *= weight


def method_ast(function):
    return ast.parse(dedent(inspect.getsource(function)))


def compile_method(tree, original, **symbols):
    namespace = {**original.__globals__, **symbols}
    ast.fix_missing_locations(tree)
    exec(compile(tree, f"<OA integration: {original.__qualname__}>", "exec"), namespace)
    return namespace[original.__name__]


def build_kf_update(original):
    """Insert BAM after the KF innovation is computed, including after ORU."""
    tree = method_ast(original)
    method = tree.body[0]
    residuals = [
        index for index, node in enumerate(method.body)
        if isinstance(node, ast.Assign)
        and any(ast.unparse(target) == "self.y" for target in node.targets)
        and isinstance(node.value, ast.BinOp) and isinstance(node.value.op, ast.Sub)
    ]
    if len(residuals) != 1:
        raise RuntimeError("Hybrid-SORT KF update changed. Could not locate the innovation.")
    method.args.kwonlyargs.append(ast.arg(arg="oa_weight"))
    method.args.kw_defaults.append(None)
    method.body.insert(residuals[0] + 1, ast.parse("damp_geometry_residual(self.y, oa_weight)").body[0])
    return compile_method(tree, original, damp_geometry_residual=damp_geometry_residual)


def build_track_update(original, kf_update):
    """Keep the raw observation in Hybrid-SORT's motion and confidence histories."""
    tree = method_ast(original)
    method = tree.body[0]
    locations = [
        node for node in ast.walk(tree) if isinstance(node, ast.Call)
        and ast.unparse(node.func) == "self.kf.update" and len(node.args) == 1
        and ast.unparse(node.args[0]) == "convert_bbox_to_z(bbox)"
    ]
    if len(locations) != 1:
        raise RuntimeError("Hybrid-SORT track update changed. Could not attach BAM.")
    method.args.kwonlyargs.append(ast.arg(arg="oa_weight"))
    method.args.kw_defaults.append(None)
    target = locations[0]
    target.func = ast.Name(id="oa_kf_update", ctx=ast.Load())
    target.args.insert(0, ast.parse("self.kf", mode="eval").body)
    target.keywords.append(ast.keyword(arg="oa_weight", value=ast.Name(id="oa_weight", ctx=ast.Load())))
    return compile_method(tree, original, oa_kf_update=kf_update)


def build_tracker_update(original):
    """Attach OAO at stage 1 and BAM at accepted stage-2 matches only."""
    tree = method_ast(original)
    high_calls = 0
    low_calls = 0
    for node in list(ast.walk(tree)):
        if not isinstance(node, ast.Call):
            continue
        called = ast.unparse(node.func)
        if called in {"associate_4_points_with_score", "associate_4_points"}:
            node.args.insert(0, node.func)
            node.func = ast.parse("self._associate_high", mode="eval").body
            high_calls += 1
        elif (called == "self.trackers[trk_ind].update" and node.args
              and ast.unparse(node.args[0]).startswith("dets_second[")):
            node.func = ast.parse("self._update_low", mode="eval").body
            node.args.insert(0, ast.Name(id="trk_ind", ctx=ast.Load()))
            node.args.append(ast.parse("trks[trk_ind, :4]", mode="eval").body)
            low_calls += 1
    if (high_calls, low_calls) != (2, 1):
        raise RuntimeError(
            f"Hybrid-SORT update changed: found {high_calls} high and {low_calls} low hooks; expected 2 and 1."
        )
    return compile_method(tree, original)


def make_oa_tracker(base, kf_class):
    """Construct a subclass using the installed Hybrid-SORT implementation."""
    original_update = build_tracker_update(base.Hybrid_Sort.update)
    bam_track_update = build_track_update(
        base.KalmanBoxTracker.update, build_kf_update(kf_class.update)
    )
    spatial_hmiou = base.ASSO_FUNCS["Height_Modulated_IoU"]

    class OASort(base.Hybrid_Sort):
        def __init__(self, *args, oa_config, **kwargs):
            super().__init__(*args, **kwargs)
            self.oa_config = oa_config
            self.oa_stats = Counter()
            if oa_config.use_bam and not self.use_byte:
                raise RuntimeError("BAM needs Hybrid-SORT's low-score matching (use_byte=True).")

        def update(self, detections, img_info, img_size):
            self.image_hw = img_info[:2]
            tracks = original_update(self, detections, img_info, img_size)
            if self.oa_config.use_bam:
                # Algorithm 5 uses matched tracks. A newly initialized track still
                # has Hybrid-SORT's negative-score last_observation sentinel.
                observed = [track for track in self.trackers if track.time_since_update == 0
                            and track.last_observation[4] >= 0]
                boxes = [track.last_observation[:4] for track in observed]
                coefficients = occlusion_coefficients(boxes, self.image_hw, self.oa_config)
                for track, coefficient in zip(observed, coefficients):
                    track.oa_last_observed = float(coefficient)
                self.oa_stats["observed_occluded_tracks"] += int(np.count_nonzero(coefficients))
            return tracks

        def _associate_high(self, original, detections, predictions, *args, **kwargs):
            config = self.oa_config
            if not config.use_oao or config.tau == 0 or not len(detections) or not len(predictions):
                return original(detections, predictions, *args, **kwargs)

            coefficients = occlusion_coefficients(predictions[:, :4], self.image_hw, config)
            raw = self.asso_func(detections, predictions)
            adjusted = occlusion_aware_similarity(raw, coefficients, config.tau)
            self.oa_stats["predicted_occluded_tracks"] += int(np.count_nonzero(coefficients))
            original_assignment = original.__globals__.get("linear_assignment")
            if original_assignment is None or "linear_assignment" not in original.__code__.co_names:
                raise RuntimeError("Hybrid-SORT stage 1 no longer calls linear_assignment.")

            def assign_with_oao(cost):
                if cost.shape != raw.shape:
                    raise RuntimeError("Unexpected shape of the stage-1 assignment matrix.")
                self.oa_stats["oao_solver_calls"] += 1
                # Cost is minimized. Swap only the spatial term, retaining the
                # track direction and confidence terms and the upstream gate.
                return original_assignment(cost + raw - adjusted)

            namespace = {**original.__globals__, "linear_assignment": assign_with_oao}
            with_oao = FunctionType(
                original.__code__, namespace, original.__name__,
                original.__defaults__, original.__closure__,
            )
            with_oao.__kwdefaults__ = original.__kwdefaults__
            return with_oao(detections, predictions, *args, **kwargs)

        def _update_low(self, track_index, detection, prediction):
            track = self.trackers[track_index]
            if not self.oa_config.use_bam:
                track.update(detection)
                return

            similarity = spatial_hmiou(detection[None, :4], prediction[None, :4])[0, 0]
            weight = bias_aware_weight(similarity, getattr(track, "oa_last_observed", 0.0))
            bam_track_update(track, detection, oa_weight=weight)
            self.oa_stats["bam_updates"] += 1
            if weight < 1:
                self.oa_stats["bam_damped_updates"] += 1

    return OASort
'''
(CODE_DIR / "oa_sort_impl.py").write_text(OA_SOURCE, encoding="utf-8")

## 6. 합성 박스로 모듈 점검

겹침 면적, 다중 가림의 중복 집계, Gaussian 가중치 범위, OAO 점수와 BAM 잔차를 작은 수치 예제로 확인한다.

In [ ]:
module_check = r"""
import numpy as np
from oa_sort_impl import (
    OAConfig, occlusion_coefficients, occlusion_aware_similarity,
    bias_aware_weight, damp_geometry_residual,
)

plain = OAConfig(gaussian=False)
boxes = [[0, 0, 10, 20], [5, 6, 15, 28]]
np.testing.assert_allclose(occlusion_coefficients(boxes, (32, 32), plain), [0.35, 0])
duplicate = [*boxes, boxes[1]]
np.testing.assert_allclose(occlusion_coefficients(duplicate, (32, 32), plain), [0.35, 0, 0])
weighted = occlusion_coefficients(boxes, (32, 32), OAConfig())
assert 0 < weighted[0] <= 0.35 and weighted[1] == 0
assert occlusion_coefficients([], (32, 32), plain).shape == (0,)
np.testing.assert_allclose(
    occlusion_aware_similarity(np.array([[0.5, 0.5]]), np.array([0.6, 0]), 0.15),
    [[0.485, 0.575]],
)
residual = np.ones((5, 1))
damp_geometry_residual(residual, bias_aware_weight(0.5, 0.6))
np.testing.assert_allclose(residual.ravel(), [0.2, 0.2, 0.2, 1, 0.2])
print("OAM / OAO / BAM synthetic checks passed.")
"""
run([PYTHON, "-c", module_check], cwd=CODE_DIR, env=BASE_ENV)

## 7. 동일한 YOLOX 검출로 추적기 실행

공식 `demo_track.py`의 모델 로딩과 추론을 사용한다. 검출 좌표를 원본 이미지 픽셀로 변환한 뒤 `shared_detections.npz`에 저장하고, 각 모드에 같은 배열을 전달한다. 검출이 비어 있는 프레임도 추적기에 전달해 트랙의 경과 시간을 계산한다. 두 모드 모두 HMIoU와 2차 저신뢰 매칭을 사용한다.

결과 요약에서 `predicted_occluded_tracks`는 고신뢰 매칭 시 가림이 추정된 트랙 수의 누적값, `oao_solver_calls`는 OAO 점수를 사용해 배정한 횟수, `bam_updates`는 저신뢰 매칭에 BAM을 적용한 횟수다. 영상에 해당 조건이 발생하지 않으면 카운트는 0일 수 있다.

In [ ]:
RUNNER_SOURCE = r'''
"""Use the official demo to capture one set of detections, then replay it."""
import argparse
import ast
from collections import Counter
from dataclasses import replace
import importlib
import inspect
import json
from pathlib import Path
import sys

import cv2
import numpy as np

from oa_sort_impl import OAConfig, make_oa_tracker


DANCE_CFG = "exps/example/mot/yolox_dancetrack_val_hybrid_sort.py"
MODE_FLAGS = {"hybrid": (False, False), "oao_only": (True, False),
              "bam_only": (False, True), "oa_sort": (True, True)}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}


def model_detections(outputs, ratio):
    """Return [x1,y1,x2,y2,score] per detection in original image pixels."""
    candidates = outputs[0]
    if candidates is None:
        return np.empty((0, 5), dtype=np.float32)
    if hasattr(candidates, "detach"):
        candidates = candidates.detach().float().cpu().numpy()
    candidates = np.asarray(candidates, dtype=np.float32)
    if candidates.ndim != 2 or candidates.shape[1] < 5:
        raise ValueError(f"Unexpected YOLOX detection tensor: {candidates.shape}")
    boxes = candidates[:, :5].copy()
    boxes[:, :4] /= ratio
    if candidates.shape[1] > 5:
        boxes[:, 4] *= candidates[:, 5]
    return boxes


class DetectionCapture:
    def __init__(self, frames):
        self.frames = frames
        self.detections = []
        self.tracker_spec = None

    def install(self, namespace):
        predictor_class = namespace.get("Predictor")
        upstream_tracker = namespace.get("Hybrid_Sort")
        if predictor_class is None or upstream_tracker is None:
            raise RuntimeError("Unexpected Hybrid-SORT demo API: expected Predictor and Hybrid_Sort.")

        def create_tracker(*args, **kwargs):
            if self.tracker_spec is not None:
                raise RuntimeError("Expected one Hybrid-SORT tracker in the official image demo.")
            self.tracker_spec = args, kwargs
            return DetectionOnlyTracker()

        original_inference = predictor_class.inference

        def capture_inference(predictor, image, *args, **kwargs):
            result = original_inference(predictor, image, *args, **kwargs)
            outputs, info = result[:2]
            index = len(self.detections)
            if index >= len(self.frames):
                raise RuntimeError("The demo received more images than expected.")
            if isinstance(image, (str, Path)) and Path(image).resolve() != self.frames[index]:
                raise RuntimeError(f"Unexpected input order at frame {index + 1}: {image}")
            if getattr(predictor, "test_size", None) is not None:
                test_h, test_w = predictor.test_size
                ratio = min(test_h / info["height"], test_w / info["width"])
            else:
                ratio = float(info["ratio"])
            if not np.isfinite(ratio) or ratio <= 0:
                raise RuntimeError(f"Invalid detector scale: {ratio}")
            self.detections.append(model_detections(outputs, ratio))
            return result

        predictor_class.inference = capture_inference
        namespace["Hybrid_Sort"] = create_tracker


class DetectionOnlyTracker:
    def update(self, *_args, **_kwargs):
        return np.empty((0, 5), dtype=np.float32)


def collect_detections(repo, frames, weights, fp16):
    """Enter the official demo with its original CLI, config and checkpoint loader."""
    checkpoint_path = weights.expanduser().resolve(strict=True)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Checkpoint is not a file: {checkpoint_path}")

    script = repo / "tools/demo_track.py"
    tree = ast.parse(script.read_text(encoding="utf-8"), filename=str(script))

    # Older FastReID uses collections.Mapping. Import it only when requested.
    lazy_reid = ast.parse(
        "def FastReIDInterface(*args, **kwargs):\n"
        "    from fast_reid.fast_reid_interfece import FastReIDInterface as Interface\n"
        "    return Interface(*args, **kwargs)\n"
    ).body[0]
    for index, node in enumerate(tree.body):
        if (isinstance(node, ast.ImportFrom)
                and node.module == "fast_reid.fast_reid_interfece"
                and [(alias.name, alias.asname) for alias in node.names]
                    == [("FastReIDInterface", None)]):
            tree.body[index] = ast.copy_location(lazy_reid, node)
            break

    guards = [
        node for node in tree.body
        if isinstance(node, ast.If) and isinstance(node.test, ast.Compare)
        and isinstance(node.test.left, ast.Name) and node.test.left.id == "__name__"
        and any(isinstance(value, ast.Constant) and value.value == "__main__"
                for value in node.test.comparators)
    ]
    if len(guards) != 1:
        raise RuntimeError("Could not locate the official Hybrid-SORT demo entry point.")
    entry = guards[0].body
    main_calls = [
        index for index, stmt in enumerate(entry)
        if isinstance(stmt, ast.Expr) and isinstance(stmt.value, ast.Call)
        and isinstance(stmt.value.func, ast.Name) and stmt.value.func.id == "main"
    ]
    if len(main_calls) != 1:
        raise RuntimeError("Could not locate the Hybrid-SORT demo main() call.")

    # The experiment config can override the CLI checkpoint before main().
    entry.insert(main_calls[0], ast.parse("args.ckpt = oa_checkpoint_path").body[0])
    entry.insert(0, ast.parse("install_detection_capture(globals())").body[0])

    # Bind the actual load to the same file even if main() rewrites ckpt_file.
    demo_main, = [node for node in tree.body
                  if isinstance(node, ast.FunctionDef) and node.name == "main"]
    checkpoint_loads = [
        node for node in ast.walk(demo_main)
        if isinstance(node, ast.Call) and ast.unparse(node.func) == "torch.load"
        and node.args and isinstance(node.args[0], ast.Name)
        and node.args[0].id == "ckpt_file"
    ]
    if len(checkpoint_loads) != 1:
        raise RuntimeError("Could not locate the Hybrid-SORT checkpoint load.")
    checkpoint_loads[0].args[0] = ast.parse("oa_checkpoint_path", mode="eval").body
    ast.fix_missing_locations(tree)

    capture = DetectionCapture(frames)
    original_argv = sys.argv
    original_path = sys.path.copy()
    # OpenCV GUI calls are unavailable in Colab.
    cv2.namedWindow = lambda *_args, **_kwargs: None
    cv2.imshow = lambda *_args, **_kwargs: None
    cv2.waitKey = lambda *_args, **_kwargs: -1
    cv2.destroyAllWindows = lambda *_args, **_kwargs: None
    # Upstream main initializes vis_folder only when --save_result is set.
    sys.argv = [str(script), "--demo_type", "image", "-f", DANCE_CFG,
                "-c", str(checkpoint_path), "--path", str(frames[0].parent), "--fuse", "--save_result"]
    if fp16:
        sys.argv.append("--fp16")
    sys.path.insert(0, str(script.parent))
    try:
        namespace = {"__name__": "__main__", "__file__": str(script),
                     "install_detection_capture": capture.install,
                     "oa_checkpoint_path": str(checkpoint_path)}
        print(f"[OA-SORT] Detector checkpoint: {checkpoint_path}", flush=True)
        exec(compile(tree, str(script), "exec"), namespace)
    finally:
        sys.argv = original_argv
        sys.path[:] = original_path
    if capture.tracker_spec is None or len(capture.detections) != len(frames):
        raise RuntimeError(
            f"Expected {len(frames)} captured frames and one tracker config, "
            f"got {len(capture.detections)} frames. Check the upstream demo log."
        )
    return capture.detections, capture.tracker_spec


def record_frame(image, tracks, mode, frame_number, frame_dir, text_file, min_box_area):
    vis = image.copy()
    cv2.putText(vis, f"{mode} | frame {frame_number}", (12, 26),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (235, 235, 235), 2, cv2.LINE_AA)
    rows = 0
    for x1, y1, x2, y2, track_id in tracks:
        width, height = x2 - x1, y2 - y1
        if not np.isfinite([x1, y1, x2, y2, track_id]).all():
            continue
        if width <= 0 or height <= 0 or width * height <= min_box_area:
            continue
        track_id = int(track_id)
        color = tuple(80 + (track_id * factor) % 176 for factor in (37, 61, 97))
        left, top, right, bottom = map(round, (x1, y1, x2, y2))
        cv2.rectangle(vis, (left, top), (right, bottom), color, 2)
        cv2.putText(vis, str(track_id), (max(0, left), max(18, top - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
        text_file.write(f"{frame_number},{track_id},{x1:.4f},{y1:.4f},"
                        f"{width:.4f},{height:.4f},1,-1,-1,-1\n")
        rows += 1
    target = frame_dir / f"{frame_number:08d}.jpg"
    if not cv2.imwrite(str(target), vis):
        raise OSError(f"Could not save {target}")
    return rows


def replay(frames, detections, tracker_spec, base, kf_class, modes, config, dest, sequence):
    tracker_args, tracker_kwargs = tracker_spec
    arguments = inspect.signature(base.Hybrid_Sort).bind(*tracker_args, **tracker_kwargs)
    arguments.apply_defaults()
    # Eq. 8 and Eq. 9 use HMIoU for this configuration. Both sides of the
    # comparison also need the same low-score association.
    arguments.arguments["asso_func"] = "Height_Modulated_IoU"
    arguments.arguments["use_byte"] = True
    tracker_args, tracker_kwargs = arguments.args, arguments.kwargs
    oa_tracker = make_oa_tracker(base, kf_class) if any(mode != "hybrid" for mode in modes) else None
    summary = {}

    for mode in modes:
        use_oao, use_bam = MODE_FLAGS[mode]
        if mode == "hybrid":
            tracker = base.Hybrid_Sort(*tracker_args, **tracker_kwargs)
        else:
            tracker = oa_tracker(
                *tracker_args, **tracker_kwargs,
                oa_config=replace(config, use_oao=use_oao, use_bam=use_bam),
            )
        min_box_area = float(getattr(tracker.args, "min_box_area", 0))
        frame_dir = dest / mode / "frames"
        frame_dir.mkdir(parents=True, exist_ok=True)
        total_rows = 0
        with (frame_dir.parent / f"{sequence}.txt").open("w", encoding="utf-8") as text_file:
            for frame_number, (frame, boxes) in enumerate(zip(frames, detections, strict=True), 1):
                image = cv2.imread(str(frame))
                if image is None:
                    raise OSError(f"Could not read {frame}")
                image_hw = image.shape[:2]
                # Detector boxes were rescaled once during capture. A unit-scale
                # tracker call preserves the same coordinates in every mode.
                tracks = tracker.update(boxes.copy(), image_hw, image_hw)
                total_rows += record_frame(
                    image, tracks, mode, frame_number, frame_dir, text_file, min_box_area,
                )

        keys = ("predicted_occluded_tracks", "observed_occluded_tracks",
                "oao_solver_calls", "bam_updates", "bam_damped_updates")
        stats = getattr(tracker, "oa_stats", Counter())
        summary[mode] = {"track_rows": total_rows, **{key: stats[key] for key in keys}}
        print(f"{mode}: {summary[mode]}", flush=True)
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--repo", required=True, type=Path)
    parser.add_argument("--frames", required=True, type=Path)
    parser.add_argument("--weights", required=True, type=Path)
    parser.add_argument("--output", required=True, type=Path)
    parser.add_argument("--sequence", required=True)
    parser.add_argument("--modes", required=True, nargs="+", choices=MODE_FLAGS)
    parser.add_argument("--tau", type=float, default=0.15)
    parser.add_argument("--depth-pixels", type=float, default=5)
    parser.add_argument("--fp16", action="store_true")
    parser.add_argument("--no-gaussian", action="store_true")
    options = parser.parse_args()

    if len(set(options.modes)) != len(options.modes):
        parser.error("--modes contains a duplicate.")
    frames = sorted(path.resolve() for path in options.frames.iterdir()
                    if path.suffix.lower() in IMAGE_SUFFIXES)
    if not frames:
        raise FileNotFoundError(f"No input images at {options.frames}")
    config = OAConfig(tau=options.tau, depth_pixels=options.depth_pixels,
                      gaussian=not options.no_gaussian)
    options.output.mkdir(parents=True, exist_ok=True)

    base = importlib.import_module("trackers.hybrid_sort_tracker.hybrid_sort")
    kf = importlib.import_module("trackers.hybrid_sort_tracker.kalmanfilter_score_new")
    detections, tracker_spec = collect_detections(options.repo, frames, options.weights, options.fp16)
    np.savez_compressed(
        options.output / "shared_detections.npz",
        frame_names=np.array([path.name for path in frames]),
        **{f"frame_{index:08d}": boxes for index, boxes in enumerate(detections, 1)},
    )
    summary = replay(
        frames, detections, tracker_spec, base, kf.KalmanFilterNew_score_new,
        options.modes, config, options.output, options.sequence,
    )
    report = {"spatial_metric": "Height_Modulated_IoU", "use_byte": True,
              "processed_frames": len(frames),
              "frames_without_detections": sum(len(boxes) == 0 for boxes in detections),
              "modes": summary}
    (options.output / "tracking_summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")


if __name__ == "__main__":
    main()
'''


runner = CODE_DIR / "oa_sort_run.py"
runner.write_text(RUNNER_SOURCE, encoding="utf-8")
import json

if not RUN_MODES:
    raise ValueError("RUN_MODES에 추적기 하나 이상을 지정하세요.")

sequence_label = SEQUENCE if INPUT_MODE == "dancetrack" else "uploaded_video"
TRACKING_DIR = RUN_DIR / "tracking"
tracking_env = BASE_ENV.copy()
if ALLOW_LEGACY_CHECKPOINT:
    tracking_env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
args = [PYTHON, runner, "--repo", REPO, "--frames", FRAMES_DIR,
        "--weights", checkpoint, "--output", TRACKING_DIR,
        "--sequence", sequence_label, "--tau", OA_TAU, "--depth-pixels", OA_DEPTH_PIXELS,
        "--modes", *RUN_MODES]
if USE_FP16:
    args.append("--fp16")
if not OA_GAUSSIAN:
    args.append("--no-gaussian")
import subprocess

log_path = RUN_DIR / "tracking_run.log"
with log_path.open("w", encoding="utf-8") as log_file:
    with subprocess.Popen(
        [str(arg) for arg in args],
        cwd=REPO,
        env={**tracking_env, "PYTHONUNBUFFERED": "1"},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
        exit_code = process.wait()

if exit_code != 0:
    raise RuntimeError(f"추적 실패 (종료 코드 {exit_code}). 상세 로그: {log_path}")

summary = json.loads((TRACKING_DIR / "tracking_summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2, ensure_ascii=False))


## 8. MP4, MOT TXT와 검출 결과 저장

각 모드의 MP4에는 프레임 번호와 track ID가 표시된다. 추적 TXT는 MOTChallenge 형식 `frame,id,x,y,w,h,conf,-1,-1,-1`이다. 공통 검출 파일에는 프레임별 `[x1,y1,x2,y2,score]` 배열이 들어 있다. 아래 셀은 화면에서 영상을 재생하고, 마지막 셀에서 실험 파일을 다운로드한다.

기본 180프레임은 동작 확인용이다. 전체 시퀀스를 처리하려면 `MAX_FRAMES=0`을 사용한다. AssA, IDF1, ID switches는 해당 프레임 범위에 맞는 정답과 TrackEval로 측정한다. 이 영상만으로 전체 DanceTrack validation 점수를 산출할 수는 없다. [DanceTrack TrackEval 안내](https://github.com/DanceTrack/DanceTrack#evaluation)

In [ ]:
from IPython.display import Video, display

reports = []
for mode in RUN_MODES:
    mode_dir = TRACKING_DIR / mode
    result_mp4 = mode_dir / f"{mode}.mp4"
    run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
         "-framerate", f"{fps:.8f}", "-start_number", "1", "-i",
         mode_dir / "frames/%08d.jpg",
         "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", "-c:v", "libx264",
         "-pix_fmt", "yuv420p", "-movflags", "+faststart", result_mp4])
    reports += [result_mp4, mode_dir / f"{sequence_label}.txt"]
    print(mode)
    display(Video(str(result_mp4), embed=True, width=760))

run_metadata = {
    "implementation": "OA-SORT (paper reimplementation), arXiv:2603.06034v2",
    "hybrid_sort_commit": REVISION,
    "tracking_config": VAL_CFG,
    "checkpoint_sha256": checkpoint_sha256,
    "checkpoint_name": checkpoint.name,
    "fp16": USE_FP16,
    "modes": list(RUN_MODES),
    "input": SEQUENCE if INPUT_MODE == "dancetrack" else video.name,
    "fps": fps,
    "oa_tau": OA_TAU,
    "oa_depth_pixels": OA_DEPTH_PIXELS,
    "oa_gaussian": OA_GAUSSIAN,
    "summary": summary,
}
run_info = RUN_DIR / "run_info.json"
run_info.write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")

In [ ]:
from google.colab import files

SAVE_TO_DRIVE = False
bundle = RUN_DIR / "oa_sort_results.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in reports + [
        run_info, TRACKING_DIR / "shared_detections.npz",
        TRACKING_DIR / "tracking_summary.json", CODE_DIR / "oa_sort_impl.py", runner,
    ]:
        archive.write(path, path.relative_to(RUN_DIR))

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/OASORT_results") / RUN_DIR.name
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(bundle, destination / bundle.name)
    print("저장 위치:", destination / bundle.name)
else:
    files.download(str(bundle))

## 9. OA-SORT Evaluation

In [ ]:
import csv
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import subprocess
import sys
from zipfile import ZipFile

import numpy as np
from IPython.display import Markdown, display

# 기본 DanceTrack 모드에서는 기존 val.zip에서 해당 시퀀스의 GT만 읽습니다.
GT_PATH = ""  # 직접 지정할 때는 예: "/content/dancetrack0079_gt.txt"
CASE_WINDOW = 60  # 자동 선택할 연속 프레임 수. 0은 자동 구간 평가 생략.
CASE_FRAMES = None  # 원본 프레임 범위 (시작, 끝), 예: DanceTrack0058의 (195, 235)
TRACKERS = ("hybrid", "oa_sort")
TRACKER_LABELS = ("Hybrid-SORT", "OA-SORT (재현)")

if CASE_WINDOW < 0:
    raise ValueError("CASE_WINDOW는 0 이상이어야 합니다.")

tracking_dir = Path(TRACKING_DIR)
evaluation_dir = Path(RUN_DIR) / "evaluation"
summary_path = tracking_dir / "tracking_summary.json"
detections_path = tracking_dir / "shared_detections.npz"
if not summary_path.is_file() or not detections_path.is_file():
    raise FileNotFoundError("Hybrid-SORT와 OA-SORT 추적 셀을 먼저 성공적으로 실행하세요.")

tracking_summary = json.loads(summary_path.read_text(encoding="utf-8"))
completed_modes = tracking_summary["modes"]
for mode in TRACKERS:
    if mode not in completed_modes:
        raise ValueError(f"추적 결과에 {mode!r} 모드가 없습니다. 두 모드를 함께 실행하세요.")
tracker_files = {mode: tracking_dir / mode / f"{sequence_label}.txt" for mode in TRACKERS}
for path in tracker_files.values():
    if not path.is_file():
        raise FileNotFoundError(path)

with np.load(detections_path, allow_pickle=False) as detections:
    frame_names = detections["frame_names"].tolist()
frame_count = len(frame_names)
if not frame_count or frame_count != int(tracking_summary["processed_frames"]):
    raise ValueError("검출 파일의 프레임 목록과 추적 결과의 프레임 수가 다릅니다.")

if INPUT_MODE == "dancetrack":
    source_frame_numbers = [int(Path(name).stem) for name in frame_names]
else:
    source_frame_numbers = list(range(1, frame_count + 1))
if any(b != a + 1 for a, b in zip(source_frame_numbers, source_frame_numbers[1:])):
    raise ValueError("공식 시퀀스 프레임이 연속이어야 합니다. 누락된 입력 프레임을 확인하세요.")

evaluation_dir.mkdir(parents=True, exist_ok=True)
sequence_member = PurePosixPath(sequence_label) / "gt" / "gt.txt"


def read_gt_from_archive(archive):
    matches = [name for name in archive.namelist()
               if PurePosixPath(name).parts[-3:] == sequence_member.parts]
    if len(matches) != 1:
        raise FileNotFoundError(f"val.zip에서 {sequence_member}를 한 개 찾을 수 없습니다.")
    return archive.read(matches[0]).decode("utf-8-sig")


def read_ground_truth():
    if GT_PATH:
        path = Path(GT_PATH).expanduser()
        return path.read_text(encoding="utf-8-sig"), str(path)
    if INPUT_MODE != "dancetrack":
        raise ValueError("직접 입력한 영상은 동일 프레임의 MOT 형식 GT_PATH를 지정하세요.")
    for raw_path in (DANCETRACK_ZIP_PATH, VAL_CACHED_ZIP):
        if raw_path and Path(raw_path).is_file():
            with ZipFile(raw_path) as archive:
                return read_gt_from_archive(archive), f"{raw_path} · {sequence_member}"
    from remotezip import RemoteZip
    try:
        with RemoteZip(VAL_ZIP_URL, support_suffix_range=False, timeout=25) as archive:
            return read_gt_from_archive(archive), f"{VAL_ZIP_URL} · {sequence_member}"
    except Exception as error:
        raise RuntimeError("GT 부분 읽기에 실패했습니다. 해당 시퀀스의 gt.txt를 준비하고 GT_PATH에 지정하세요.") from error


gt_text, gt_source = read_ground_truth()
gt_frame_map = {original: index for index, original in enumerate(source_frame_numbers, 1)}
gt_by_frame = [[] for _ in frame_names]
for row in csv.reader(gt_text.splitlines()):
    if not row:
        continue
    if len(row) < 8:
        raise ValueError(f"GT는 MOT 형식의 8개 이상 필드가 필요합니다. 문제 행 = {row}")
    position = gt_frame_map.get(int(row[0]))
    if position is not None:
        gt_by_frame[position - 1].append(row)

if not any(int(row[6]) != 0 for rows in gt_by_frame for row in rows):
    raise ValueError("처리한 프레임에 활성 GT 객체가 없습니다. GT 파일과 입력 시퀀스를 확인하세요.")


def mot_window(rows, start, stop):
    return ([local_frame, *row[1:]]
            for local_frame, frame_rows in enumerate(rows[start:stop], 1)
            for row in frame_rows)


def write_rows(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        csv.writer(file, lineterminator="\n").writerows(rows)


predictions = {}
for mode, path in tracker_files.items():
    rows = [[] for _ in frame_names]
    with path.open(encoding="utf-8", newline="") as file:
        for row in csv.reader(file):
            if not row:
                continue
            number = int(row[0])
            if not 1 <= number <= frame_count:
                raise ValueError(f"{mode} 추적 결과 프레임 {number}가 평가 범위를 벗어납니다.")
            rows[number - 1].append(row)
    predictions[mode] = rows

# 평가 패키지는 별도 경로에 설치해 기존 추적 가상환경의 의존성을 유지합니다.
evaluator_version = "1.0.0"
eval_lib = Path(ROOT) / f"eval_packages_{evaluator_version}"
eval_env = {
    **os.environ,
    "PYTHONPATH": os.pathsep.join(map(str, (eval_lib, CODE_DIR))),
}
if not (eval_lib / "trackeval" / "__init__.py").is_file():
    uv_executable = shutil.which("uv") or str(Path(sys.executable).with_name("uv"))
    subprocess.run(
        [uv_executable, "pip", "install", "--python", str(PYTHON),
         "--target", str(eval_lib), "--no-deps", f"trackeval=={evaluator_version}"],
        check=True,
    )

# 케이스 구간을 고르기 위해 GT 박스만으로 가림 면적 비율을 계산합니다.
occlusion_worker = r"""
import csv
import json
from pathlib import Path
import sys

import cv2
import numpy as np
from oa_sort_impl import OAConfig, occlusion_coefficients

gt_file, image_file, count, depth = sys.argv[1:]
image = cv2.imread(image_file)
if image is None:
    raise FileNotFoundError(image_file)
rows = [[] for _ in range(int(count))]
with Path(gt_file).open(encoding="utf-8") as file:
    for row in csv.reader(file):
        if int(row[6]) != 0 and int(row[7]) == 1:
            rows[int(row[0]) - 1].append(row)
config = OAConfig(depth_pixels=float(depth), gaussian=False)
frame_stats = []
for frame in rows:
    boxes = np.asarray([[float(v) for v in row[2:6]] for row in frame]).reshape(-1, 4)
    boxes[:, 2:] += boxes[:, :2]
    covered = occlusion_coefficients(boxes, image.shape[:2], config)
    frame_stats.append([len(covered), int((covered > 0.75).sum()), float(covered.sum())])
print(json.dumps(frame_stats))
"""

depth_pixels = float(OA_DEPTH_PIXELS)
scopes = {"processed": (0, frame_count)}
case_info = None

if CASE_FRAMES is not None or (0 < CASE_WINDOW < frame_count):
    gt_for_case = evaluation_dir / "gt_for_occlusion.txt"
    write_rows(gt_for_case, mot_window(gt_by_frame, 0, frame_count))
    output = subprocess.check_output(
        [str(PYTHON), "-c", occlusion_worker, str(gt_for_case),
         str(Path(FRAMES_DIR) / frame_names[0]), str(frame_count), str(depth_pixels)],
        cwd=CODE_DIR, env=eval_env, text=True,
    )
    stats = np.asarray(json.loads(output), dtype=float)
    if CASE_FRAMES is not None:
        first, last = CASE_FRAMES
        if first not in gt_frame_map or last not in gt_frame_map or first > last:
            raise ValueError("CASE_FRAMES를 추적 결과에 포함된 원본 프레임 범위로 지정하세요.")
        start, stop = gt_frame_map[first] - 1, gt_frame_map[last]
    else:
        total, severe, covered = [
            np.convolve(stats[:, column], np.ones(CASE_WINDOW), "valid")
            for column in range(3)
        ]
        score = np.divide(severe, total, out=np.zeros_like(total), where=total > 0)
        mean = np.divide(covered, total, out=np.zeros_like(total), where=total > 0)
        # 심한 가림 비율 우선, 동점은 평균 가림 면적 비율 우선입니다.
        start = int(np.lexsort((-np.arange(len(score)), mean, score))[-1])
        stop = start + CASE_WINDOW

    objects, severe_objects, coverage = stats[start:stop].sum(axis=0)
    if objects and (coverage or CASE_FRAMES is not None):
        scopes["case"] = (start, stop)
        case_info = {
            "frames": [source_frame_numbers[start], source_frame_numbers[stop - 1]],
            "object_frames": int(objects),
            "severe_occlusion_share_pct": 100 * severe_objects / objects,
            "mean_box_occlusion_pct": 100 * coverage / objects,
            "depth_px": depth_pixels,
            "severe_threshold": 0.75,
        }
    else:
        print("지정 구간에 GT 가림 사례가 없어 전체 지표만 계산합니다.")

eval_worker = r"""
import json
from pathlib import Path
import sys

import numpy as np
import trackeval

work = Path(sys.argv[1])
sequence, first, second, length = sys.argv[2:]
modes = (first, second)
dataset_cfg = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
dataset_cfg.update(
    GT_FOLDER=str(work / "gt"),
    TRACKERS_FOLDER=str(work / "trackers"),
    OUTPUT_FOLDER=str(work / "trackeval"),
    TRACKERS_TO_EVAL=list(modes),
    TRACKER_SUB_FOLDER="data",
    CLASSES_TO_EVAL=["pedestrian"],
    BENCHMARK="DanceTrack",
    SPLIT_TO_EVAL="val",
    SKIP_SPLIT_FOL=True,
    DO_PREPROC=False,
    SEQ_INFO={sequence: int(length)},
    PRINT_CONFIG=False,
)
eval_cfg = trackeval.Evaluator.get_default_eval_config()
eval_cfg.update(
    USE_PARALLEL=False, PRINT_CONFIG=False, PRINT_RESULTS=False,
    OUTPUT_SUMMARY=False, OUTPUT_DETAILED=False, PLOT_CURVES=False,
    PRINT_ONLY_COMBINED=True, TIME_PROGRESS=False,
)
dataset = trackeval.datasets.MotChallenge2DBox(dataset_cfg)
metrics = [trackeval.metrics.HOTA(),
           trackeval.metrics.CLEAR({"THRESHOLD": 0.5, "PRINT_CONFIG": False}),
           trackeval.metrics.Identity({"THRESHOLD": 0.5, "PRINT_CONFIG": False})]
results, messages = trackeval.Evaluator(eval_cfg).evaluate([dataset], metrics)
dataset_name = dataset.get_name()
report = {}
for mode in modes:
    combined = results[dataset_name][mode]["COMBINED_SEQ"]["pedestrian"]
    hota, clear, identity = (combined[k] for k in ("HOTA", "CLEAR", "Identity"))
    report[mode] = {
        "HOTA": 100 * float(np.mean(hota["HOTA"])),
        "AssA": 100 * float(np.mean(hota["AssA"])),
        "DetA": 100 * float(np.mean(hota["DetA"])),
        "IDF1": 100 * float(identity["IDF1"]),
        "MOTA": 100 * float(clear["MOTA"]),
        "IDSW": int(clear["IDSW"]),
        "FP": int(clear["CLR_FP"]),
        "FN": int(clear["CLR_FN"]),
        "Frag": int(clear["Frag"]),
    }
(work / "metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
"""

worker_path = evaluation_dir / "evaluate_trackeval.py"
worker_path.write_text(eval_worker, encoding="utf-8")
metric_order = ("HOTA", "AssA", "IDF1", "MOTA", "DetA", "IDSW", "FP", "FN", "Frag")
count_metrics = {"IDSW", "FP", "FN", "Frag"}
comparisons = {}
out_rows = [("구간", "원본 시작", "원본 끝", "지표", *TRACKER_LABELS,
             "OA-Hybrid", "단위", "우수")]


def comparison_table(metrics, names, labels=TRACKER_LABELS):
    head = ["| 지표 | " + " | ".join(labels) + " | OA − Hybrid | 우수 |",
            "|---|---:|---:|---:|---|"]
    rows = []
    for metric in names:
        left, right = (metrics[mode][metric] for mode in TRACKERS)
        delta = right - left
        count = metric in count_metrics
        tied = round(delta, 2) == 0
        winner = "동률" if tied else (
            labels[1] if (delta < 0 if count else delta > 0) else labels[0]
        )
        unit = "건" if count else "%p"
        left_text = str(left) if count else f"{left:.2f}%"
        right_text = str(right) if count else f"{right:.2f}%"
        change = f"{int(delta):+d}" if count else f"{0.0 if tied else delta:+.2f}"
        head.append(f"| {metric} {'↓' if count else '↑'} | {left_text} | {right_text} | {change} {unit} | {winner} |")
        rows.append((metric, left, right, delta, unit, winner))
    display(Markdown("\n".join(head)))
    return rows


for scope, (start, stop) in scopes.items():
    work = evaluation_dir / scope
    write_rows(work / "gt" / sequence_label / "gt" / "gt.txt", mot_window(gt_by_frame, start, stop))
    for mode in TRACKERS:
        write_rows(work / "trackers" / mode / "data" / f"{sequence_label}.txt",
                   mot_window(predictions[mode], start, stop))
    if not any(int(row[6]) != 0 for rows in gt_by_frame[start:stop] for row in rows):
        print(f"{scope} 구간에 활성 GT 객체가 없어 지표를 계산하지 않았습니다.")
        continue
    subprocess.run(
        [str(PYTHON), str(worker_path), str(work), sequence_label, *TRACKERS, str(stop - start)],
        env=eval_env, cwd=evaluation_dir, check=True,
    )
    scores = json.loads((work / "metrics.json").read_text(encoding="utf-8"))
    comparisons[scope] = scores
    label = "처리한 전체 구간" if scope == "processed" else "가림 사례 구간"
    start_frame, end_frame = source_frame_numbers[start], source_frame_numbers[stop - 1]
    display(Markdown(f"### {sequence_label} · {label} · {start_frame}–{end_frame}프레임"))
    if scope == "case":
        print(f"가림 면적 75% 초과 비중: {case_info['severe_occlusion_share_pct']:.2f}% "
              f"| 평균 가림 면적 비율: {case_info['mean_box_occlusion_pct']:.2f}%")
    for row in comparison_table(scores, metric_order):
        out_rows.append((label, start_frame, end_frame, *row))

# 논문 기준값은 현재 실행 결과와 별도 표로 표시합니다.
paper_val = {
    "hybrid": {"HOTA": 59.39, "AssA": 44.90, "IDF1": 60.67},
    "oa_sort": {"HOTA": 61.49, "AssA": 48.03, "IDF1": 63.65},
}
display(Markdown("### 논문 참고값 · DanceTrack validation 전체 · 표 11"))
comparison_table(paper_val, ("HOTA", "AssA", "IDF1"), ("Hybrid-SORT", "OA-SORT (논문)"))
display(Markdown(
    "위 논문값은 validation 전체 결과입니다. 현재 셀의 표는 저장된 추적 프레임 구간만 평가합니다. "
    "[OA-SORT 논문 Table 11](https://arxiv.org/html/2603.06034v2) · "
    "[DanceTrack 공식 평가 안내](https://github.com/DanceTrack/DanceTrack#evaluation)"
))

write_rows(evaluation_dir / "oa_sort_metrics.csv", out_rows)
(evaluation_dir / "oa_sort_metrics.json").write_text(json.dumps(
    {"sequence": sequence_label, "frames": source_frame_numbers, "gt": gt_source,
     "trackeval_version": evaluator_version, "case": case_info, "results": comparisons,
     "paper_dancetrack_val_table11": paper_val},
    indent=2, ensure_ascii=False,
), encoding="utf-8")
print("결과 CSV", evaluation_dir / "oa_sort_metrics.csv")

In [ ]:
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

root = Path(TRACKING_DIR)
stats = json.loads((root / "tracking_summary.json").read_text(encoding="utf-8"))
print("OA-SORT 실행 통계:", stats["modes"]["oa_sort"])

def rows_by_frame(path):
    frames = defaultdict(Counter)
    with path.open(newline="", encoding="utf-8") as f:
        for frame, track_id, x, y, w, h, *_ in csv.reader(f):
            frames[int(frame)][(int(track_id), *map(float, (x, y, w, h)))] += 1
    return frames

hybrid = rows_by_frame(root / "hybrid" / f"{sequence_label}.txt")
oa = rows_by_frame(root / "oa_sort" / f"{sequence_label}.txt")

different = sorted(
    frame for frame in hybrid.keys() | oa.keys()
    if hybrid[frame] != oa[frame]
)
print("박스 또는 ID가 다른 프레임 수:", len(different))
print("처음 다른 프레임:", different[:20])